In [1]:
import sys
sys.path.append("..")

from ERA_Distribution_Classes_Python.Classes.ERADist import ERADist
from ERA_Distribution_Classes_Python.Classes.ERANataf import ERANataf
from ERA_Distribution_Classes_Python.Classes.FORM_HLRF import FORM_HLRF
from ERA_Distribution_Classes_Python.Classes.FORM_fmincon import FORM_fmincon
from ERA_Distribution_Classes_Python.Classes.SuS import SuS

In [2]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt 
from dsm_core.structure import Structure
from dsm_core.solver_1st import Solver1stOrder
from measures_of_nonlinearity import kappa_1, kappa_2, kappa_12, r1, r2
from syst_1_model_functions import t_S_linear, t_S_hyperplane_linear

## Material Properties

In [3]:
# modified IPE 120 -> eta = 100% for linear hyperplane
E = 210e6 # kN/m2 
A = 1.321e-3 # m2
I = 2.7784576742780014e-06 # m4
h = 0.12  # m
z = h/2  # m
alpha = 1.14

In [4]:
def t_R(M_k, I=I, z=z, alpha=alpha):
    """
    Takes in the Steel Bending Strength M_k (random variable) in kN/cm2
    I in m^4
    z in m
    alpha is the plastic ratio 

    Returns the characteristic Bending Moment Resistance M_c_Rk for the given system in kNm
    """
    return I/z * M_k * 100e2 * alpha

## System Definition

In [5]:
# Vectorized Version of the Structural response function (Better for array handling later)
t_S_vectorized = np.vectorize(t_S_hyperplane_linear, otypes=[float])

## Target characteristic values for calibrating random variables

In [ ]:
s_k = 1.1 # snow load on ground kN/m2
q_b = 0.65 # wind pressure kN/m2
w_k = q_b * 0.8 # wind load kN/m2 with c_pe,10 = 0.8 (Area D)
m_k = 35.5  # material strength kN/cm2

## Random Variables

In [7]:
# Snow time-invariant part
mu_Theta_1 = 0.81
cov_Theta_1 = 0.26
sig_Theta_1 = mu_Theta_1 * cov_Theta_1
Theta_L1 = ERADist('lognormal','MOM',[mu_Theta_1, sig_Theta_1])

# Snow load on ground
mu_L1 = 1.0
cov_L1 = 0.2
sig_L1 = mu_L1 * cov_L1
L1 = ERADist('gumbel','MOM',[mu_L1,sig_L1])

In [8]:
percentile_L1 = L1.icdf(0.98)
print(f"Snow 98% Percentile: {percentile_L1}")

Snow 98% Percentile: 1.5184551765313796


In [9]:
# Wind time-invariant part
mu_Theta_2 = 0.97
cov_Theta_2 = 0.26
sig_Theta_2 = mu_Theta_2 * cov_Theta_2
Theta_L2 = ERADist('lognormal','MOM',[mu_Theta_2, sig_Theta_2])

# Wind velocity pressure
mu_L2 = 1.0 
cov_L2 = 0.14
sig_L2 = mu_L2 * cov_L2
L2 = ERADist('gumbel','MOM',[mu_L2, sig_L2])

In [10]:
percentile_L2 = L2.icdf(0.98)
print(f"Wind 98% Percentile: {percentile_L2}")

Wind 98% Percentile: 1.3629186235719657


In [11]:
# Structural Response Model Uncertainty (from JCSS Probabilistic Model Code, Part 3, Table 3.9.1)
mu_Theta_S = 1.0
cov_Theta_S = 0.1
sig_Theta_S = mu_Theta_S * cov_Theta_S
Theta_S = ERADist('lognormal','MOM',[mu_Theta_S, sig_Theta_S])  # Distribution for Moments in frames

In [12]:
# Steel bending model uncertainty
mu_Theta_M = 1.15
cov_Theta_M = 0.05
sig_Theta_M = mu_Theta_M * cov_Theta_M
Theta_M = ERADist('lognormal','MOM',[mu_Theta_M, sig_Theta_M])

# Steel yielding strength
mu_M = 1.0
cov_M = 0.05
sig_M = mu_M * cov_M
M = ERADist('lognormal','MOM',[mu_M, sig_M])

In [13]:
percentile_M = M.icdf(0.05)
print(f"Steel 5% Percentile: {percentile_M}")

Steel 5% Percentile: 0.9199464756612658


## Shifting / Scaling Random Variables 

In [14]:
# Snow Load on Ground, shifted to characteristic value
snow_shift = s_k / percentile_L1 # ratio of target to current percentile, by which mean and std get multiplied

mu_L1_shifted = mu_L1 * snow_shift
sig_L1_shifted = sig_L1 * snow_shift
L1_shifted = ERADist('gumbel','MOM',[mu_L1_shifted, sig_L1_shifted])

print(f"""Snow Load on Ground gets shifted by {snow_shift}""")
print(f"""Old mean: {mu_L1}; New mean: {mu_L1_shifted}""")
print(f"""Old std: {sig_L1}; New std: {sig_L1_shifted}""")
print(f"""Old 98th percentile: {L1.icdf(.98)}; New 98th percentile: {L1_shifted.icdf(.98)}""")
print(f"""Old COV: {L1.std()/L1.mean()}; New COV: {L1_shifted.std()/L1_shifted.mean()}""")

Snow Load on Ground gets shifted by 0.7244204616646898
Old mean: 1.0; New mean: 0.7244204616646898
Old std: 0.2; New std: 0.14488409233293795
Old 98th percentile: 1.5184551765313796; New 98th percentile: 1.0999999999999999
Old COV: 0.19999999999999998; New COV: 0.19999999999999996


In [15]:
# Wind velocity pressure, shifted to characteristic value
wind_shift = q_b / percentile_L2

mu_L2_shifted = mu_L2 * wind_shift
sig_L2_shifted = sig_L2 * wind_shift
L2_shifted = ERADist('gumbel','MOM',[mu_L2_shifted, sig_L2_shifted])

print(f"""Wind velocity pressure gets shifted by {wind_shift}""")
print(f"""Old mean: {mu_L2}; New mean: {mu_L2_shifted}""")
print(f"""Old std: {sig_L2}; New std: {sig_L2_shifted}""")
print(f"""Old 98th percentile: {L2.icdf(.98)}; New 98th percentile: {L2_shifted.icdf(.98)}""")
print(f"""Old COV: {L2.std()/L2.mean()}; New COV: {L2_shifted.std()/L2_shifted.mean()}""")

Wind velocity pressure gets shifted by 0.4769176888173018
Old mean: 1.0; New mean: 0.4769176888173018
Old std: 0.14; New std: 0.06676847643442226
Old 98th percentile: 1.3629186235719657; New 98th percentile: 0.65
Old COV: 0.14; New COV: 0.13999999999999999


In [16]:
# Steel bending resistance, shifted to characteristic value
steel_shift = m_k / percentile_M

mu_M_shifted = mu_M * steel_shift
sig_M_shifted = sig_M * steel_shift
M_shifted = ERADist('lognormal','MOM',[mu_M_shifted, sig_M_shifted])

print(f"""Steel bending resistance gets shifted by {steel_shift}""")
print(f"""Old mean: {mu_M}; New mean: {mu_M_shifted}""")
print(f"""Old std: {sig_M}; New std: {sig_M_shifted}""")
print(f"""Old 5th percentile: {M.icdf(0.05)}; New 5th percentile: {M_shifted.icdf(0.05)}""")
print(f"""Old COV: {M.std()/M.mean()}; New COV: {M_shifted.std()/M_shifted.mean()}""")

Steel bending resistance gets shifted by 38.58920158858403
Old mean: 1.0; New mean: 38.58920158858403
Old std: 0.05; New std: 1.9294600794292016
Old 5th percentile: 0.9199464756612658; New 5th percentile: 35.5
Old COV: 0.04999999999999947; New COV: 0.04999999999999946


## Characteristic values derived from Random Variables for further calculation

In [17]:
l_1k = L1_shifted.icdf(0.98)
l_2k = L2_shifted.icdf(0.98)
m_k = M_shifted.icdf(0.05)
print(f"l_1k = {l_1k:.4f} kN/m^2")
print(f"l_2k = {l_2k:.4f} kN/m^2")
print(f"m_k = {m_k:.4f} kN/cm^2")

l_1k = 1.1000 kN/m^2
l_2k = 0.6500 kN/m^2
m_k = 35.5000 kN/cm^2


## Partial Safety Factors

In [18]:
gamma_M = 1.0  # Resistance
gamma_F1 = 1.5   # Snow Load
gamma_F2 = 1.5   # Wind Load
#psi_0 = 1.0 # 0.6     # Windload

## Design Values

In [19]:
l_1d = s_k * gamma_F1
l_2d = q_b * gamma_F2 
m_d = m_k / gamma_M
print(f"l_1d = {l_1d:.4f} kN/m^2")
print(f"l_2d = {l_2d:.4f} kN/m^2")
print(f"m_d = {m_d:.4f} kN/cm^2")

l_1d = 1.6500 kN/m^2
l_2d = 0.9750 kN/m^2
m_d = 35.5000 kN/cm^2


## Measures of Nonlinearity

In [20]:
## Measures of Nonlinearity
k1 = kappa_1(l_1k=l_1k, l_1d=l_1d, t_S=t_S_hyperplane_linear)
k2 = kappa_2(l_2k=l_2k, l_2d=l_2d, t_S=t_S_hyperplane_linear)
k12 = kappa_12(l_1k=l_1k, l_1d=l_1d, l_2k=l_2k, l_2d=l_2d, t_S=t_S_hyperplane_linear)
r1 = r1(l_1k=l_1k, l_2k=l_2k, t_S=t_S_hyperplane_linear)
r2 = r2(l_1k=l_1k, l_2k=l_2k, t_S=t_S_hyperplane_linear)


print(f"kappa1 = {k1}")
print(f"kappa2 = {k2}")
print(f"kappa12 = {k12}")
print(f"r1 = {r1}")
print(f"r2 = {r2}")

kappa1 = 1.0000000000000002
kappa2 = 0.9999999999999999
kappa12 = 0.9999999999999996
r1 = 0.12181262640583582
r2 = 0.8781873735941641


## Design parameters p for option 1 and 2 

In [21]:
# Design Opt 1
e_d_1 = t_S_hyperplane_linear(l_1d, l_2d)

# Design Opt 2
argument_1 = gamma_F1 * t_S_hyperplane_linear(l_1k,  (gamma_F2 / gamma_F1) * l_2k)
argument_2 = gamma_F2 * t_S_hyperplane_linear((gamma_F1 / gamma_F2) * l_1k, l_2k)
e_d_2 = max(argument_1, argument_2)

p_opt1 = gamma_M * e_d_1 / t_R(M_k=m_d)
p_opt2 = gamma_M * e_d_2 / t_R(M_k=m_d)

print("Design Option 1:")
print(f"e_d = {e_d_1} kNm")
print(f"p_opt1 = {p_opt1}")
print(f"\n")
print("Design Option 2:")
print(f"e_d = {e_d_2} kNm")
print(f"p_opt2 = {p_opt2}")

Design Option 1:
e_d = 18.74071611437746 kNm
p_opt1 = 1.0000010192455662


Design Option 2:
e_d = 18.74071611437746 kNm
p_opt2 = 1.0000010192455662


## Construction of the Nataf Distribution

In [22]:
# Array of marginal distributions
marginal_dist_with_uncertainties = [Theta_M, M_shifted, Theta_L1, L1_shifted, Theta_L2, L2_shifted, Theta_S]

# Correlation matrix (no correlation yet)
dimensions = len(marginal_dist_with_uncertainties)
R_xx = np.eye(dimensions)

# Construction of the Nataf Distribution
nataf_with_uncertainties = ERANataf(M=marginal_dist_with_uncertainties, Correlation=R_xx)

In [23]:
# Array of marginal distributions
marginal_dist_without_uncertainties = [M_shifted, L1_shifted, L2_shifted]

# Correlation matrix (no correlation yet)
dimensions = len(marginal_dist_without_uncertainties)
R_xx = np.eye(dimensions)

# Construction of the Nataf Distribution
nataf_without_uncertainties = ERANataf(M=marginal_dist_without_uncertainties, Correlation=R_xx)

## Subset Simulation with model uncertainties

### Design Option 1

In [24]:
# deterministic design action effect
e_d_opt1 = t_S_hyperplane_linear(l_1=gamma_F1 * s_k, l_2=gamma_F2 * q_b) # kNm

In [25]:
def g_opt_1_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [26]:
# %% Samples Return
samples_return = 1
# %% subset simulation
N  = 10000        # Total number of samples for each level
p0 = 0.1         # Probability of each subset, chosen adaptively

print('\n\nSUBSET SIMULATION: ')
[Pf_SuS_1, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_1_sus, nataf_with_uncertainties, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  10.958897438496003
	*aCS lambda = 0.7680200456261327 	*aCS sigma = 0.7680200456261327 	*aCS accrate = 0.44478114478114483

-Threshold intermediate level  1  =  6.533329176043718
	*aCS lambda = 0.5533642792381269 	*aCS sigma = 0.5533642792381269 	*aCS accrate = 0.4371492704826038

-Threshold intermediate level  2  =  2.1241783675756674
	*aCS lambda = 0.4429630628589259 	*aCS sigma = 0.4429630628589259 	*aCS accrate = 0.4377104377104377

-Threshold intermediate level  3  =  0.0
	*aCS lambda = 0.3919555028188042 	*aCS sigma = 0.3919555028188042 	*aCS accrate = 0.4365243004418262


In [27]:
print("Subset Simulation for Design Option 1")
print(f"P(F) = {Pf_SuS_1}")
X = sp.stats.Normal()
beta = - X.icdf(Pf_SuS_1)
print(f"beta = {beta}")
print(samplesX)

Subset Simulation for Design Option 1
P(F) = 0.00034600000000000006
beta = 3.392729248891055
[array([[ 1.0877403 , 36.10118295,  0.8400738 , ...,  1.32140222,
         0.77663439,  1.14636165],
       [ 1.0877403 , 36.10118295,  0.8400738 , ...,  1.32140222,
         0.77663439,  1.14636165],
       [ 1.0877403 , 36.10118295,  0.8400738 , ...,  1.32140222,
         0.77663439,  1.14636165],
       ...,
       [ 1.12921199, 38.49425408,  0.5708118 , ...,  2.24602388,
         0.57591436,  1.16309184],
       [ 1.12475503, 36.78718909,  0.7654771 , ...,  1.85295138,
         0.77041699,  1.07410678],
       [ 1.12029634, 35.79488405,  0.82015507, ...,  1.70275082,
         0.8404098 ,  1.07721179]], shape=(10000, 7))]


---

### Design Option 2

In [28]:
# deterministic design action effect
argument_1 = gamma_F1 * t_S_hyperplane_linear(l_1= s_k, l_2= (gamma_F2 / gamma_F1) * q_b)
argument_2 = gamma_F2 * t_S_hyperplane_linear(l_1= (gamma_F1 / gamma_F2) * s_k, l_2= q_b)
e_d_opt2 = max(argument_1, argument_2) # kNm

In [29]:
def g_opt_2_sus(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [30]:
# %% Samples Return
samples_return = 1
# %% subset simulation
N  = 10000        # Total number of samples for each level
p0 = 0.1         # Probability of each subset, chosen adaptively

print('\n\nSUBSET SIMULATION: ')
[Pf_SuS_2, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_2_sus, nataf_with_uncertainties, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  10.843235066286065
	*aCS lambda = 0.739087456414221 	*aCS sigma = 0.739087456414221 	*aCS accrate = 0.44534231200897867

-Threshold intermediate level  1  =  6.434189207055758
	*aCS lambda = 0.5434455557724986 	*aCS sigma = 0.5434455557724986 	*aCS accrate = 0.4389450056116722

-Threshold intermediate level  2  =  2.1013135978053343
	*aCS lambda = 0.430983137798149 	*aCS sigma = 0.430983137798149 	*aCS accrate = 0.43726150392817065

-Threshold intermediate level  3  =  0.0
	*aCS lambda = 0.4202164916222265 	*aCS sigma = 0.4202164916222265 	*aCS accrate = 0.44192677070828335


In [31]:
print("Subset Simulation for Design Option 2")
print(f"P(F) = {Pf_SuS_2}")

X = sp.stats.Normal()
beta = - X.icdf(Pf_SuS_2)
print(f"beta = {beta}")
# print(samplesX)

Subset Simulation for Design Option 2
P(F) = 0.0003383000000000001
beta = 3.398889687027763


## Subset Simulation without model uncertainties

### Design Option 1

In [32]:
# deterministic design action effect
e_d_opt1 = t_S_hyperplane_linear(l_1=gamma_F1 * s_k, l_2=gamma_F2 * q_b) # kNm

In [33]:
def g_opt_1_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: M = Steel yield strength
    x[1]: L1 = Snow Load on Ground
    x[2]: L2 = Wind velocity pressure
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * t_R(M_k=x[:,0])
    action_side = t_S_vectorized(l_1=(x[:,1]), l_2=(x[:,2]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [34]:
# %% Samples Return
samples_return = 1
# %% subset simulation
N  = 10000        # Total number of samples for each level
p0 = 0.1         # Probability of each subset, chosen adaptively

print('\n\nSUBSET SIMULATION: ')
[Pf_SuS_1, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_1_sus, nataf_without_uncertainties, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  9.36908218775284
	*aCS lambda = 0.730034210341255 	*aCS sigma = 0.730034210341255 	*aCS accrate = 0.44051627384960723

-Threshold intermediate level  1  =  7.263270423660744
	*aCS lambda = 0.5283789865606672 	*aCS sigma = 0.5283789865606672 	*aCS accrate = 0.43535353535353527

-Threshold intermediate level  2  =  5.2578005019072425
	*aCS lambda = 0.4371952620565772 	*aCS sigma = 0.4371952620565772 	*aCS accrate = 0.4360269360269361

-Threshold intermediate level  3  =  3.1529520424345745
	*aCS lambda = 0.37806773529084353 	*aCS sigma = 0.37806773529084353 	*aCS accrate = 0.4344556677890011

-Threshold intermediate level  4  =  1.0442448326078726
	*aCS lambda = 0.33230633822393996 	*aCS sigma = 0.33230633822393996 	*aCS accrate = 0.4396184062850728

-Threshold intermediate level  5  =  0.0
	*aCS lambda = 0.30797174249306386 	*aCS sigma = 0.30797174249306386 	*aCS accrate = 0.4371241408934708

In [35]:
print("Subset Simulation for Design Option 1")
print(f"P(F) = {Pf_SuS_1}")
X = sp.stats.Normal()
beta = - X.icdf(Pf_SuS_1)
print(f"beta = {beta}")
print(samplesX)

Subset Simulation for Design Option 1
P(F) = 3.1480000000000015e-06
beta = 4.516197324862368
[array([[34.51801021,  0.75010523,  1.1839671 ],
       [35.49488221,  0.76004882,  1.13150104],
       [34.9067782 ,  0.62818643,  1.1456989 ],
       ...,
       [37.38962404,  0.87464441,  1.2104135 ],
       [36.86033328,  0.90811034,  1.18551456],
       [36.96776339,  0.91461743,  1.1347817 ]], shape=(10000, 3))]


---

### Design Option 2

In [36]:
# deterministic design action effect
argument_1 = gamma_F1 * t_S_hyperplane_linear(l_1= s_k, l_2= (gamma_F2 / gamma_F1) * q_b)
argument_2 = gamma_F2 * t_S_hyperplane_linear(l_1= (gamma_F1 / gamma_F2) * s_k, l_2= q_b)
e_d_opt2 = max(argument_1, argument_2) # kNm

In [37]:
def g_opt_2_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: M = Steel yield strength
    x[1]: L1 = Snow Load on Ground
    x[2]: L2 = Wind velocity pressure
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * t_R(M_k=x[:,0])
    action_side = t_S_vectorized(l_1=(x[:,1]), l_2=(x[:,2]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [38]:
# %% Samples Return
samples_return = 1
# %% subset simulation
N  = 10000        # Total number of samples for each level
p0 = 0.1         # Probability of each subset, chosen adaptively

print('\n\nSUBSET SIMULATION: ')
[Pf_SuS_2, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_2_sus, nataf_without_uncertainties, samples_return)



SUBSET SIMULATION: 
Evaluating performance function:	OK!

-Threshold intermediate level  0  =  9.382786552107751
	*aCS lambda = 0.7536077470385026 	*aCS sigma = 0.7536077470385026 	*aCS accrate = 0.44511784511784513

-Threshold intermediate level  1  =  7.245214745964362
	*aCS lambda = 0.5367280566795251 	*aCS sigma = 0.5367280566795251 	*aCS accrate = 0.43692480359147023

-Threshold intermediate level  2  =  5.1403127242180044
	*aCS lambda = 0.43239921018607913 	*aCS sigma = 0.43239921018607913 	*aCS accrate = 0.43456790123456784

-Threshold intermediate level  3  =  3.068313238724145
	*aCS lambda = 0.3778425171410543 	*aCS sigma = 0.3778425171410543 	*aCS accrate = 0.439057239057239

-Threshold intermediate level  4  =  1.0357932774631486
	*aCS lambda = 0.32550042205409774 	*aCS sigma = 0.32550042205409774 	*aCS accrate = 0.43636363636363634

-Threshold intermediate level  5  =  0.0
	*aCS lambda = 0.32485238066425565 	*aCS sigma = 0.32485238066425565 	*aCS accrate = 0.4424189814814

In [39]:
print("Subset Simulation for Design Option 2")
print(f"P(F) = {Pf_SuS_2}")

X = sp.stats.Normal()
beta = - X.icdf(Pf_SuS_2)
print(f"beta = {beta}")
# print(samplesX)

Subset Simulation for Design Option 2
P(F) = 2.9160000000000014e-06
beta = 4.532390172029068


## FORM Analysis with model uncertainties

### Design Option 1

In [40]:
def g_opt_1_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [41]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_1, dg=[], distr=nataf, sensitivity_analysis=0, u0=0)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_FORM, dg=[], distr=nataf_with_uncertainties)


*scipy.optimize.minimize() with  SLSQP  Method

  11  iterations... Reliability index =  3.4451629852993326  --- Failure probability =  0.00028535757728182626 




In [42]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_1_FORM(x_star)}")

u_star = [-0.52007413 -0.52007408  0.10491421  0.07711938  2.55875387  1.9195536
  1.03890909]
x_star = [ 1.11910138 37.55237301  0.80525623  0.71078479  1.80626453  0.63330471
  1.10368813]
(alpha_2)^2 = [2.27882487e-02 2.27882443e-02 9.27361627e-04 5.01080505e-04
 5.51616793e-01 3.10442335e-01 9.09359372e-02]
beta = 3.4451629852993326
P(F) = 0.00028535757728182626
g(X*) = -3.258793945803973e-07


### Design Option 2

In [43]:
def g_opt_2_FORM(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [44]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_2, dg=[], distr=nataf, sensitivity_analysis=0, u0=1)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_FORM, dg=[], distr=nataf_with_uncertainties)


*scipy.optimize.minimize() with  SLSQP  Method

  11  iterations... Reliability index =  3.4451629852993326  --- Failure probability =  0.00028535757728182626 




In [45]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_2_FORM(x_star)}")

u_star = [-0.52007413 -0.52007408  0.10491421  0.07711938  2.55875387  1.9195536
  1.03890909]
x_star = [ 1.11910138 37.55237301  0.80525623  0.71078479  1.80626453  0.63330471
  1.10368813]
(alpha_2)^2 = [2.27882487e-02 2.27882443e-02 9.27361627e-04 5.01080505e-04
 5.51616793e-01 3.10442335e-01 9.09359372e-02]
beta = 3.4451629852993326
P(F) = 0.00028535757728182626
g(X*) = -3.258793945803973e-07


## FORM Analysis without model uncertainties

### Design Option 1

In [46]:
def g_opt_1_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: M = Steel yield strength
    x[1]: L1 = Snow Load on Ground
    x[2]: L2 = Wind velocity pressure
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * t_R(M_k=x[0])
    action_side = t_S_vectorized(l_1=(x[1]), l_2=(x[2]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [47]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_1, dg=[], distr=nataf, sensitivity_analysis=0, u0=0)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_FORM, dg=[], distr=nataf_without_uncertainties)


*scipy.optimize.minimize() with  SLSQP  Method

  11  iterations... Reliability index =  4.5518928229212205  --- Failure probability =  2.6582709546293572e-06 




In [48]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_1_FORM(x_star)}")

u_star = [-1.04878369  0.21090094  4.42439847]
x_star = [36.57326802  0.72909917  1.08404699]
(alpha_2)^2 = [0.05308695 0.00214671 0.94476634]
beta = 4.5518928229212205
P(F) = 2.6582709546293572e-06
g(X*) = 4.882483395363124e-08


### Design Option 2

In [49]:
def g_opt_2_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: M = Steel yield strength
    x[1]: L1 = Snow Load on Ground
    x[2]: L2 = Wind velocity pressure
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * t_R(M_k=x[0])
    action_side = t_S_vectorized(l_1=(x[1]), l_2=(x[2]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [50]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_2, dg=[], distr=nataf, sensitivity_analysis=0, u0=1)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_FORM, dg=[], distr=nataf_without_uncertainties)


*scipy.optimize.minimize() with  SLSQP  Method

  11  iterations... Reliability index =  4.5518928229212205  --- Failure probability =  2.6582709546293572e-06 




In [51]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_2_FORM(x_star)}")

u_star = [-1.04878369  0.21090094  4.42439847]
x_star = [36.57326802  0.72909917  1.08404699]
(alpha_2)^2 = [0.05308695 0.00214671 0.94476634]
beta = 4.5518928229212205
P(F) = 2.6582709546293572e-06
g(X*) = 4.882483395363124e-08
